In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import shap

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
oxford_data_feat = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/Antibodies_detection/codes/crop_based/analysis/slide_features.csv")

In [7]:
numeric_cols = ['area_mean_mature_lb',
       'area_mean_pre_lb', 'area_mean_neg_lb', 'area_std_mature_lb',
       'area_std_pre_lb', 'area_std_neg_lb', 'perimeter_mean_mature_lb',
       'perimeter_mean_pre_lb', 'perimeter_mean_neg_lb',
       'perimeter_std_mature_lb', 'perimeter_std_pre_lb',
       'perimeter_std_neg_lb', 'eccentricity_mean_mature_lb',
       'eccentricity_mean_pre_lb', 'eccentricity_mean_neg_lb',
       'eccentricity_std_mature_lb', 'eccentricity_std_pre_lb',
       'eccentricity_std_neg_lb', 'major_axis_length_mean_mature_lb',
       'major_axis_length_mean_pre_lb', 'major_axis_length_mean_neg_lb',
       'major_axis_length_std_mature_lb', 'major_axis_length_std_pre_lb',
       'major_axis_length_std_neg_lb', 'minor_axis_length_mean_mature_lb',
       'minor_axis_length_mean_pre_lb', 'minor_axis_length_mean_neg_lb',
       'minor_axis_length_std_mature_lb', 'minor_axis_length_std_pre_lb',
       'minor_axis_length_std_neg_lb', 'solidity_mean_mature_lb',
       'solidity_mean_pre_lb', 'solidity_mean_neg_lb',
       'solidity_std_mature_lb', 'solidity_std_pre_lb', 'solidity_std_neg_lb',
       'extent_mean_mature_lb', 'extent_mean_pre_lb', 'extent_mean_neg_lb',
       'extent_std_mature_lb', 'extent_std_pre_lb', 'extent_std_neg_lb',
       'aspect_ratio_mean_mature_lb', 'aspect_ratio_mean_pre_lb',
       'aspect_ratio_mean_neg_lb', 'aspect_ratio_std_mature_lb',
       'aspect_ratio_std_pre_lb', 'aspect_ratio_std_neg_lb',
       'circularity_mean_mature_lb', 'circularity_mean_pre_lb',
       'circularity_mean_neg_lb', 'circularity_std_mature_lb',
       'circularity_std_pre_lb', 'circularity_std_neg_lb',
       'lb_count_mature_lb', 'lb_count_pre_lb', 'lb_count_neg_lb','brain_region_Amygdala', 'brain_region_EntCx',
       'brain_region_Striatum', 'anitibody_C110-115', 'anitibody_C34-45']

In [8]:
oxford_data_feat.drop(["Unnamed: 0"],axis=1, inplace=True)
categorical_cols = ['brain_region', 'anitibody']  # update with your columns
oxford_data_feat.drop(columns=["pat_id","LBD_flag","Brain_bank"], axis=1, inplace=True)
oxford_data_feat.drop(columns=["slide_id"], axis=1, inplace=True)
df_encoded = pd.get_dummies(oxford_data_feat, columns=categorical_cols, drop_first=False)
df_encoded = df_encoded.astype(np.float32)  # Add this line
df_encoded = df_encoded.fillna(0)
# Standardize numeric columns
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

In [52]:
df  = df_encoded[df_encoded.columns[1:]]
# Assuming df is your DataFrame
corr_matrix = df.corr().abs()
# Upper triangle of the correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# Find features with correlation > threshold
threshold = 0.75
to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
print(to_drop)

['area_std_mature_lb', 'area_std_pre_lb', 'area_std_neg_lb', 'perimeter_mean_mature_lb', 'perimeter_mean_pre_lb', 'perimeter_mean_neg_lb', 'perimeter_std_mature_lb', 'perimeter_std_pre_lb', 'perimeter_std_neg_lb', 'eccentricity_mean_mature_lb', 'eccentricity_mean_pre_lb', 'eccentricity_mean_neg_lb', 'eccentricity_std_mature_lb', 'eccentricity_std_pre_lb', 'eccentricity_std_neg_lb', 'major_axis_length_mean_mature_lb', 'major_axis_length_mean_pre_lb', 'major_axis_length_mean_neg_lb', 'major_axis_length_std_mature_lb', 'major_axis_length_std_pre_lb', 'major_axis_length_std_neg_lb', 'minor_axis_length_mean_mature_lb', 'minor_axis_length_mean_pre_lb', 'minor_axis_length_mean_neg_lb', 'minor_axis_length_std_mature_lb', 'minor_axis_length_std_pre_lb', 'minor_axis_length_std_neg_lb', 'solidity_mean_mature_lb', 'solidity_mean_pre_lb', 'solidity_mean_neg_lb', 'solidity_std_mature_lb', 'solidity_std_pre_lb', 'solidity_std_neg_lb', 'extent_mean_mature_lb', 'extent_mean_pre_lb', 'extent_mean_neg_lb

In [54]:
df_reduced = df.drop(columns=to_drop[:-1])

In [9]:
X = df_encoded[numeric_cols]

In [10]:
target_col = ['label']
y = df_encoded[target_col]

In [11]:
# Set up K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
feature_importances = np.zeros(X.shape[1])


In [12]:
skf.split(X, y)

<generator object _BaseKFold.split at 0x7efbb4789f90>

In [16]:
# Train and collect importances
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(fold)
    model = RandomForestClassifier(n_estimators=50, random_state=fold)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    feature_importances += model.feature_importances_
    y_pred = model.predict(X.iloc[val_idx])
    acc = accuracy_score(y.iloc[val_idx], y_pred)
    print("test accuracy", acc)
    

1
test accuracy 1.0
2
test accuracy 0.8571428571428571
3
test accuracy 1.0
4
test accuracy 1.0
5


/tmp/ipykernel_1029309/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029309/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029309/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029309/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029309/2

test accuracy 0.9230769230769231


In [17]:
feature_importances.shape

(62,)

In [18]:
feature_importances

array([0.16608563, 0.11576925, 0.15199844, 0.15846829, 0.75398953,
       0.12619913, 0.09274271, 0.09829979, 0.13271599, 0.0558636 ,
       0.37560066, 0.09705285, 0.3382186 , 0.99744975, 0.16586832,
       0.18202676, 0.8180229 , 0.03350875, 0.11598109, 0.11919514,
       0.10421916, 0.13678737, 0.2011449 , 0.10635149, 0.1825977 ,
       0.17758911, 0.21112977, 0.11130165, 1.41813926, 0.12363734,
       0.49964421, 0.22505866, 0.43912098, 0.27993354, 0.19407321,
       1.39998502, 0.23698332, 1.05020942, 0.44850949, 0.46026469,
       0.61988868, 0.2305375 , 0.16785806, 0.87036255, 0.09942451,
       0.1135899 , 0.34467846, 0.06492465, 0.09088312, 1.76433978,
       0.0951697 , 0.08788253, 1.42783628, 0.10600594, 0.21899448,
       0.18687459, 0.27523734, 0.01303704, 0.00889334, 0.02730462,
       0.03260245, 0.05193704])

In [19]:
len(df_encoded.columns)

63

In [20]:
df  =pd.DataFrame({"feature_name": X.columns.values, "feature_importances": feature_importances})

In [23]:
df.sort_values(by = "feature_importances", ascending=False)[:20]

,feature_name,feature_importances
49,circularity_mean_pre_lb,1.764340
52,circularity_std_pre_lb,1.427836
28,minor_axis_length_std_pre_lb,1.418139
35,solidity_std_neg_lb,1.399985
37,extent_mean_pre_lb,1.050209
13,eccentricity_mean_pre_lb,0.997450
43,aspect_ratio_mean_pre_lb,0.870363
16,eccentricity_std_pre_lb,0.818023
4,area_std_pre_lb,0.753990
40,extent_std_pre_lb,0.619889


In [81]:
X = df_reduced
y = df_encoded[target_col]

# Set up K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
feature_importances = np.zeros(X.shape[1])

In [82]:
# Train and collect importances
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(fold)
    model = RandomForestClassifier(n_estimators=50, random_state=fold)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    feature_importances += model.feature_importances_
    y_pred = model.predict(X.iloc[val_idx])
    acc = accuracy_score(y.iloc[val_idx], y_pred)
    print("test accuracy", acc)
    

1
test accuracy 0.7142857142857143
2
test accuracy 0.7857142857142857
3
test accuracy 0.8571428571428571
4
test accuracy 0.8461538461538461
5


/tmp/ipykernel_1029093/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/296093397.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/2

test accuracy 0.7692307692307693


In [5]:
(0.785+0.857+0.846)/3

0.8293333333333334

In [83]:
df  =pd.DataFrame({"feature_name": X.columns.values, "feature_importances": feature_importances})

In [84]:
df.sort_values(by = "feature_importances", ascending=False)

,feature_name,feature_importances
4,lb_count_mature_lb,0.961067
5,lb_count_pre_lb,0.852385
6,lb_count_neg_lb,0.678725
2,area_mean_neg_lb,0.555666
1,area_mean_pre_lb,0.490252
0,area_mean_mature_lb,0.487264
3,aspect_ratio_mean_neg_lb,0.437291
11,anitibody_C34-45,0.160642
10,anitibody_C110-115,0.136209
7,brain_region_Amygdala,0.120708


In [96]:
importance_matrix = np.zeros(X.shape[1])

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    model = RandomForestClassifier(n_estimators=50, random_state=fold)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X.iloc[val_idx])
    print(shap_values.shape)
    # Sum mean absolute SHAP values for the positive class (or use both)
    shap_importance = np.abs(shap_values[1]).mean(axis=1)
    importance_matrix += shap_importance

# Average over folds
importance_matrix /= skf.get_n_splits()

# Show top features
shap_df = pd.DataFrame({
    'Feature': X.columns,
    'SHAP Importance': importance_matrix
}).sort_values(by='SHAP Importance', ascending=False)

print(shap_df.head(10))

/tmp/ipykernel_1029093/69585908.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/69585908.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/69585908.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/69585908.py:5: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X.iloc[train_idx], y.iloc[train_idx])
/tmp/ipykernel_1029093/69585

(14, 12, 2)
(14, 12, 2)
(14, 12, 2)
(13, 12, 2)
(13, 12, 2)
                     Feature  SHAP Importance
4         lb_count_mature_lb         0.069677
6            lb_count_neg_lb         0.067125
2           area_mean_neg_lb         0.064215
5            lb_count_pre_lb         0.061377
0        area_mean_mature_lb         0.057612
1           area_mean_pre_lb         0.024179
11          anitibody_C34-45         0.023651
7      brain_region_Amygdala         0.020901
3   aspect_ratio_mean_neg_lb         0.019908
10        anitibody_C110-115         0.016777


In [3]:
oxford_data_feat

,Unnamed: 0,slide_id,brain_region,pat_id,anitibody,LBD_flag,label,Brain_bank,area_mean_mature_lb,area_mean_pre_lb,...,aspect_ratio_std_neg_lb,circularity_mean_mature_lb,circularity_mean_pre_lb,circularity_mean_neg_lb,circularity_std_mature_lb,circularity_std_pre_lb,circularity_std_neg_lb,lb_count_mature_lb,lb_count_pre_lb,lb_count_neg_lb
0,0,PD134-DLB--Amygdala-C110-115.svs,Amygdala,PD134,C110-115,DLB,1,Oxford,547.247741,235.936411,...,1.058582,1.008514,1.152042,1.016600,2.588069,2.857844,2.082594,18261.0,17393.0,18154.0
1,1,PD134-DLB--EntCx-C110-115.svs,EntCx,PD134,C110-115,DLB,1,Oxford,476.007864,220.410873,...,1.076915,1.054892,1.157449,1.033833,2.769480,2.948588,2.273730,10427.0,14384.0,14846.0
2,2,PD134-DLB--EntCx-C34-45.svs,EntCx,PD134,C34-45,DLB,1,Oxford,450.217087,208.978953,...,1.006494,1.061775,1.115091,1.020566,2.774461,2.653749,2.435939,8439.0,19908.0,5309.0
3,3,PD134-DLB-Striatum-C34-45.svs,Striatum,PD134,C34-45,DLB,1,Oxford,562.717987,171.851720,...,1.103462,0.988267,1.231444,1.288466,2.633916,3.088231,3.331519,7333.0,5348.0,976.0
4,4,PD152-DLB--Striatum-C110-115.svs,Striatum,PD152,C110-115,DLB,1,Oxford,474.907346,233.468092,...,1.004978,1.069451,1.173538,0.934586,2.814075,3.077795,1.685971,6357.0,12583.0,17395.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,63,PD159-C34-45-Striatum.svs,Striatum,PD159,C34-45,PDD,0,Oxford,519.612270,145.788194,...,1.196599,1.057595,1.234994,1.074923,2.766453,3.148655,2.441094,15289.0,7115.0,4656.0
64,64,PD17_Amygdala_C110-115 David Menassa.svs,Amygdala,PD17,C110-115,PDD,0,Oxford,276.863220,145.194395,...,1.009478,1.183493,1.148301,1.041042,3.222246,2.608423,2.258673,11778.0,15844.0,62828.0
65,65,PD171_C110-115_Amygdala David Menassa.svs,Amygdala,PD171,C110-115,PDD,0,Oxford,485.343504,229.983094,...,1.164588,0.972773,1.014796,1.048767,2.471793,2.208335,2.454327,8390.0,21649.0,9191.0
66,66,PD171_C110-115_Amygdala.svs,Amygdala,PD171,C110-115,PDD,0,Oxford,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
